# 03 - Couche Gold : Agrégations Métier et KPIs

Ce notebook lit les données Silver (nettoyées et enrichies) et produit
des indicateurs clés de performance (KPIs) directement exploitables
par les équipes métier d'Enedis.

Tables Gold produites :
- kpi_conso_par_region       : Consommation totale, moyenne et par site, par région et par an
- kpi_tendance_nationale     : Évolution nationale annuelle avec variation d'une année sur l'autre
- kpi_categorie_distribution : Répartition des volumes par catégorie de consommation
- kpi_top_regions            : Classement des régions les plus consommatrices

## Configuration et imports

In [0]:
# Imports des fonctions d'agregation et de fenetrage PySpark

from pyspark.sql.functions import (
    col,                      
    sum as spark_sum,         
    avg,                      
    count,                    
    round as spark_round,     
    rank,                     
    desc,               
    lit,                    
    when,                
    lag,                      # Fonction de fenetrage : recupere la valeur de la ligne précédente
    current_timestamp         
)

from pyspark.sql.window import Window                   # Une Window definit le perimetre sur lequel s'applique la fonction : partition et tri

from pyspark.sql.types import DoubleType                # Type virgule flottante double precision

DATASET_NAME = "conso_inf36_region"

print("Configuration Gold chargée")

Configuration Gold chargée


## Chargement des données Silver

In [0]:

df_silver = spark.read.table(f"silver_{DATASET_NAME}")

print(f"Lignes Silver chargees : {df_silver.count():,}")
print(f"Colonnes disponibles : {df_silver.columns}")

df_silver.limit(3).display()

Lignes Silver chargees : 15,000
Colonnes disponibles : ['horodate', 'region', 'code_region', 'profil', 'plage_de_puissance_souscrite', 'nb_points_soutirage', 'total_energie_soutiree_wh', 'courbe_moyenne_ndeg1_wh', 'indice_representativite_courbe_ndeg1', 'courbe_moyenne_ndeg2_wh', 'indice_representativite_courbe_ndeg2', 'courbe_moyenne_ndeg1_ndeg2_wh', 'indice_representativite_courbe_ndeg1_ndeg2', 'jour_max_du_mois_0_1', 'semaine_max_du_mois_0_1', 'categorie_consommation', 'score_carbone_normalise', '_silver_timestamp', '_silver_date', '_rgpd_min_sites']


horodate,region,code_region,profil,plage_de_puissance_souscrite,nb_points_soutirage,total_energie_soutiree_wh,courbe_moyenne_ndeg1_wh,indice_representativite_courbe_ndeg1,courbe_moyenne_ndeg2_wh,indice_representativite_courbe_ndeg2,courbe_moyenne_ndeg1_ndeg2_wh,indice_representativite_courbe_ndeg1_ndeg2,jour_max_du_mois_0_1,semaine_max_du_mois_0_1,categorie_consommation,score_carbone_normalise,_silver_timestamp,_silver_date,_rgpd_min_sites
2023-12-31T23:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P4: ]9-12] kVA,2231,4170809.0,1828.0,41,1485.0,42,1653.0,84,0,0,TRES ELEVEE,37.4,2026-04-29T16:43:06.112Z,2026-04-29,5
2024-06-30T22:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P4: ]9-12] kVA,2122,6447151.0,918.0,43,1485.0,43,1200.0,87,0,0,TRES ELEVEE,60.8,2026-04-29T16:43:06.112Z,2026-04-29,5
2023-12-31T23:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P6: ]15-18] kVA,1055,2435911.0,2038.0,39,2057.0,42,2048.0,81,0,0,TRES ELEVEE,46.2,2026-04-29T16:43:06.112Z,2026-04-29,5


In [0]:
# Diagnostic : affichage des vraies colonnes de la table Silver

print("=== COLONNES REELLES DE LA TABLE SILVER ===")
print() 
for c in df_silver.columns:
    print(f"  - {c}")

=== COLONNES REELLES DE LA TABLE SILVER ===

  - horodate
  - region
  - code_region
  - profil
  - plage_de_puissance_souscrite
  - nb_points_soutirage
  - total_energie_soutiree_wh
  - courbe_moyenne_ndeg1_wh
  - indice_representativite_courbe_ndeg1
  - courbe_moyenne_ndeg2_wh
  - indice_representativite_courbe_ndeg2
  - courbe_moyenne_ndeg1_ndeg2_wh
  - indice_representativite_courbe_ndeg1_ndeg2
  - jour_max_du_mois_0_1
  - semaine_max_du_mois_0_1
  - categorie_consommation
  - score_carbone_normalise
  - _silver_timestamp
  - _silver_date
  - _rgpd_min_sites


## Détection dynamique des colonnes

In [0]:
from pyspark.sql.functions import year, to_timestamp

df_silver = df_silver.withColumn("annee", year(to_timestamp(col("horodate"))))

region_col    = "region"                    
annee_col     = "annee"                     
conso_col     = "total_energie_soutiree_wh" 
sites_col     = "nb_points_soutirage"       
categorie_col = "categorie_consommation"    

print(f"region_col  = {region_col}")
print(f"annee_col   = {annee_col}")
print(f"conso_col   = {conso_col}")
print(f"sites_col   = {sites_col}")

region_col  = region
annee_col   = annee
conso_col   = total_energie_soutiree_wh
sites_col   = nb_points_soutirage


## KPI 1 - Consommation par région et par année

In [0]:
# Ce KPI répond à la question : "Combien chaque région consomme-t-elle par an ?"

if region_col and annee_col and conso_col:

    # Construction de la liste des expressions d'agrégation. On prépare les calculs que l'on va appliquer dans le groupBy.

    agg_exprs = [
        spark_sum(conso_col).alias("conso_totale_mwh"),  
        avg(conso_col).alias("conso_moyenne_mwh"),    
        count("*").alias("nb_enregistrements")             
    ]

    if sites_col:
        agg_exprs.append(
            spark_sum(sites_col).alias("nb_sites_total")  # Total des sites de la région sur l'année
        )

    df_kpi_region = (
        df_silver
        .groupBy(region_col, annee_col)  
        .agg(*agg_exprs)                 
    
        .withColumn("conso_totale_mwh",   spark_round(col("conso_totale_mwh"),  2))
        .withColumn("conso_moyenne_mwh",  spark_round(col("conso_moyenne_mwh"), 2))
    )

    # Calcul de la consommation par site (indicateur d'intensité énergétique)
    if sites_col:
        df_kpi_region = df_kpi_region.withColumn(
            "conso_par_site_mwh",
            spark_round(
                when(
                    col("nb_sites_total") > 0,                                       # Évite la division par zéro
                    col("conso_totale_mwh") / col("nb_sites_total")                  
                ).otherwise(None),  # Si nb_sites_total est 0 ou null, la valeur est nulle
                2  
            )
        )

    # Tri par région puis par année pour une lecture ordonnée
    df_kpi_region = df_kpi_region.orderBy(region_col, annee_col)

    print("KPI 1 - Consommation par région et par année :")
    df_kpi_region.display()

KPI 1 - Consommation par région et par année :


region,annee,conso_totale_mwh,conso_moyenne_mwh,nb_enregistrements,nb_sites_total,conso_par_site_mwh
Auvergne-Rhône-Alpes,2023,1.5607610927E10,3.25158561E7,492,58689542,265.94
Auvergne-Rhône-Alpes,2024,2.3587072199E10,3.685480031E7,656,78848720,299.14
Auvergne-Rhône-Alpes,2025,1.4376073184E10,2.995015247E7,492,59486642,241.67
Bourgogne-Franche-Comté,2023,5.527912021E9,1.256343641E7,492,20706848,266.96
Bourgogne-Franche-Comté,2024,8.732701034E9,1.475118418E7,656,27713848,315.1
Bourgogne-Franche-Comté,2025,5.299836043E9,1.193656766E7,492,20841128,254.3
Bretagne,2023,7.427831134E9,1.68814344E7,492,26349946,281.89
Bretagne,2024,1.0964422969E10,1.852098474E7,656,35456142,309.24
Bretagne,2025,6.144775461E9,1.383958437E7,492,26783130,229.43
Centre-Val de Loire,2023,5.13046373E9,1.155509849E7,492,17309242,296.4


## Interpretation - KPI 1

Avec 15 000 enregistrements, 12 régions métropolitaines sont représentées sur 3 années (2023, 2024, 2025) soit 36 combinaisons région × année.

Auvergne–Rhône-Alpes reste la région la plus consommatrice, avec 23,5 milliards de Wh en 2024 soit environ trois fois la consommation de la Bretagne ou du Grand Est sur la même année. Cet écart, à mon avis, reflète son poids industriel et démographique.

La hausse systématique de la consommation en 2024 par rapport à 2023 est visible dans toutes les régions. Le nombre d’enregistrements passe de 492 à 656 en 2024 ce qui suggère que l’API retourne davantage de points de mesure pour cette année — probablement en raison d’un découpage temporel plus fin ou d’une couverture réseau plus complète.

L’indicateur conso_par_site_mwh met en évidence des disparités régionales intéressantes : Centre‑Val de Loire affiche la valeur la plus élevée en 2024 (350 Wh/site). 

A mon avis, cela peut s’expliquer par une proportion plus importante de chauffage électrique dans l’habitat rural par opposition aux grandes agglomérations où la diversité des modes de chauffage est plus forte.

## KPI 2 - Évolution temporelle nationale avec variation annuelle (YoY)

In [0]:
# Ce KPI répond à : "La consommation nationale augmente-t-elle ou baisse-t-elle d'une année sur l'autre ?"
# La variation YoY (Year over Year) est un indicateur clé de la transition énergétique

if annee_col and conso_col:

    # Agrégation nationale : on regroupe tout par année (toutes régions confondues)
    df_kpi_tendance = (
        df_silver
        .groupBy(annee_col)
        .agg(
            spark_sum(conso_col).alias("conso_nationale_mwh"),  # Consommation totale nationale en MWh
            count("*").alias("nb_points_mesure")                # Nombre de points de mesure contribuant
        )

        # Conversion de MWh en TWh pour une meilleure lisibilité (1 TWh = 1 000 000 MWh)
        .withColumn(
            "conso_nationale_twh",
            spark_round(col("conso_nationale_mwh") / 1_000_000, 4)  # Division et arrondi à 4 décimales
        )
        .orderBy(annee_col)  # Tri chronologique pour que le lag() fonctionne correctement
    )

    # --- Calcul de la variation Year-over-Year avec une Window Function ---

    window_yoy = Window.orderBy(annee_col)

    df_kpi_tendance = (
        df_kpi_tendance

        .withColumn(
            "conso_annee_precedente_twh",
            lag("conso_nationale_twh", 1).over(window_yoy)                      # lag("conso_nationale_twh", 1) récupère la valeur de la ligne PRÉCÉDENTE dans la fenêtre. La première année n'a pas de prédécesseur : lag retourne null.
        )

        # Calcul de la variation en pourcentage :
        # variation = (valeur_courante - valeur_précédente) / valeur_précédente * 100
        .withColumn(
            "variation_yoy_pct",
            spark_round(
                when(
                    col("conso_annee_precedente_twh").isNotNull() &                # On ne calcule que si la valeur précédente existe et est non nulle
                    (col("conso_annee_precedente_twh") != 0),

                    # Formule de la variation en pourcentage
                    (col("conso_nationale_twh") - col("conso_annee_precedente_twh"))
                    / col("conso_annee_precedente_twh") * 100
                )
                .otherwise(None),  # Pas de variation calculable pour la première année
                2  # Arrondi à 2 décimales
            )
        )

        # Suppression de la colonne intermédiaire (plus besoin après le calcul)
        .drop("conso_annee_precedente_twh")
    )

    print("KPI 2 - Évolution nationale et variation Year-over-Year :")
    df_kpi_tendance.display()

KPI 2 - Évolution nationale et variation Year-over-Year :


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


annee,conso_nationale_mwh,nb_points_mesure,conso_nationale_twh,variation_yoy_pct
2023,8.904763905E10,4500,89047.6391,null
2024,1.34190987629E11,6000,134190.9876,50.7
2025,8.2175437864E10,4500,82175.4379,-38.76


## Interpétation - KPI 2


Sur 15 000 enregistrements couvrant 12 régions, la consommation nationale agrégée atteint 89047 TWh en 2023, 134190 TWh en 2024 (+50,7%) puis 82175 TWh en 2025 (–38,76%).

La hausse observée en 2024 est cohérente avec les constats du KPI 1 : le nombre de points de mesure passe de 4500 à 6000, ce qui indique que l’API fournit une granularité temporelle plus fine ou une couverture réseau plus complète pour cette année. Une partie de la variation YoY reflète donc la structure du dataset et non uniquement une évolution réelle de la consommation sur le réseau.


## KPI 3 - Repartition des volumes par categorie de consommation

In [0]:
# Ce KPI exploite la colonne "categorie_consommation" produite par l'UDF Python du notebook 02.
# Si cette colonne est absente, on utilise "profil" comme catégorie de substitution.
# Il répond à : "Quelle part du volume total est consommée par chaque catégorie ?"

# Détermination de la colonne catégorie à utiliser
if "categorie_consommation" in df_silver.columns:
    cat_col = "categorie_consommation"
elif "profil" in df_silver.columns:                 # profil contient les types de consommateurs (ENT1, ENT2, ENT3...) définis par Enedis
    cat_col = "profil"
    print("Info : categorie_consommation absente, utilisation de 'profil' comme substitut")
else:
    cat_col = None

if cat_col and conso_col:

    window_total = Window.partitionBy()  # partitionBy() sans argument = une seule partition globale

    df_kpi_categorie = (
        df_silver
        .groupBy(cat_col)
        .agg(
            count("*").alias("nb_mesures"),                       # Nombre de mesures pour cette catégorie
            spark_sum(conso_col).alias("conso_totale_mwh")        # Consommation totale de la catégorie
        )
        .withColumn("conso_totale_mwh", spark_round(col("conso_totale_mwh"), 2))

        # Calcul du pourcentage de volume que représente cette catégorie sur le total national.
        # spark_sum("conso_totale_mwh").over(window_total) calcule la somme de TOUTES les catégories.

        .withColumn(
            "pct_volume_total",
            spark_round(
                col("conso_totale_mwh") /
                spark_sum("conso_totale_mwh").over(window_total) * 100,
                2
            )
        )

        .orderBy(desc("conso_totale_mwh"))
    )

    print("KPI 3 - Répartition par catégorie de consommation :")
    df_kpi_categorie.display()

else:
    print("SKIP KPI 3 : aucune colonne catégorie disponible")

KPI 3 - Répartition par catégorie de consommation :


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


categorie_consommation,nb_mesures,conso_totale_mwh,pct_volume_total
TRES ELEVEE,13542,3.05391758464E11,99.99
ELEVEE,348,2.2228657E7,0.01
MODEREE,4,77422.0,0.0
INCONNU,1106,null,null


## Interpretation - KPI 3

La colonne categorie_consommation est produite par l’UDF Python du notebook 02 qui classe chaque ligne selon son volume de consommation en trois niveaux : MODÉRÉE, ÉLEVÉE ou TRÈS ÉLEVÉE.

La quasi‑totalité des enregistrements (99,99 %) se retrouve dans la catégorie TRÈS ÉLEVÉE. Cela s’explique par le fait que l’UDF a été conçue pour catégoriser des consommations individuelles (compteurs résidentiels exprimés en kWh) alors que les données Enedis utilisées ici sont des agrégats régionaux. Chaque ligne représente la somme de milliers de points de soutirage sur une région entière ce qui génère mécaniquement des volumes en milliards de Wh automatiquement classés dans la catégorie la plus élevée.

La catégorie INCONNU (1 106 mesures avec consommation nulle) correspond aux lignes où la valeur total_energie_soutiree_wh est nulle ou absente dans la source empêchant l’UDF d’attribuer une catégorie.


## KPI 4 - Classement des régions les plus consommatrices

In [0]:
# Ce KPI répond à : "Quelles régions consomment le plus pour la dernière année disponible ?"
# Utilité : priorisation des investissements réseau, planification capacitaire

if region_col and conso_col and annee_col:

    derniere_annee = df_silver.agg({annee_col: "max"}).collect()[0][0]
    print(f"Classement pour l'année : {derniere_annee}")

    # Définition de la fenêtre pour le classement :

    window_rank = Window.orderBy(desc("conso_totale_mwh"))                    # trie les régions de la plus consommatrice à la moins consommatrice.

    df_kpi_top_regions = (
        df_silver

        .filter(col(annee_col) == derniere_annee)
        .groupBy(region_col)
        .agg(
            spark_sum(conso_col).alias("conso_totale_mwh"),
            spark_sum(sites_col).alias("nb_sites") if sites_col else lit(None).alias("nb_sites")
        )

        .withColumn("conso_totale_mwh", spark_round(col("conso_totale_mwh"), 2))

        .withColumn("classement", rank().over(window_rank))             # rank() attribue un numéro de classement selon la fenêtre définie. La région avec la plus grande consommation reçoit le rang 1. En cas d'égalité, les deux régions reçoivent le même rang (ex : 1, 1, 3...)

        .orderBy("classement")
    )

    print(f"KPI 4 - Classement des régions ({derniere_annee}) :")
    df_kpi_top_regions.display()

Classement pour l'année : 2025
KPI 4 - Classement des régions (2025) :


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


region,conso_totale_mwh,nb_sites,classement
Auvergne-Rhône-Alpes,1.4376073184E10,59486642,1
Île-de-France,9.705650658E9,44646460,2
Hauts-de-France,8.971314372E9,36273036,3
Grand-Est,7.384764966E9,27944796,4
Occitanie,6.406207689E9,24301912,5
Bretagne,6.144775461E9,26783130,6
Provence-Alpes-Côte d'Azur,6.012634378E9,21567932,7
Nouvelle Aquitaine,5.972554126E9,22335239,8
Bourgogne-Franche-Comté,5.299836043E9,20841128,9
Centre-Val de Loire,4.866886463E9,17450416,10


## Interpretation - KPI 4

Le classement 2025 couvre 12 des 13 régions métropolitaines françaises. La seule région absente est vraisemblablement la Corse dont la faible population, à mon avis, génère trop peu de lignes dans le dataset avec limit = 15000.

Auvergne–Rhône-Alpes confirme sa position de leader avec 14,3 milliards de Wh soit environ quatre fois la consommation de la Normandie (rang 12). Cet écart reflète les différences de tissu industriel, de densité de population et de profil de consommation entre les régions.

L’Île‑de‑France se positionne au rang 2 avec 9,7 milliards de Wh un résultat beaucoup plus cohérent avec son poids démographique (12 millions d’habitants).

La fonction rank() pour gérer automatiquement les ex‑æquo en attribuant le même rang à deux régions ayant une consommation identique.

J’ai voulu intégrer ce KPI car il constitue selon moi un indicateur directement exploitable pour prioriser les investissements réseau et orienter la planification capacitaire régionale en identifiant clairement les zones les plus consommatrices et donc les plus critiques pour le dimensionnement du réseau.

## Écriture des tables Gold en Delta Lake

In [0]:
# Dictionnaire associant chaque nom de table Gold au DataFrame correspondant.
# On utilise dir() pour vérifier si le DataFrame a bien été créé (évite une NameError).

gold_tables = {
    "kpi_conso_par_region":       df_kpi_region      if "df_kpi_region"      in dir() else None,
    "kpi_tendance_nationale":     df_kpi_tendance    if "df_kpi_tendance"    in dir() else None,
    "kpi_categorie_distribution": df_kpi_categorie   if "df_kpi_categorie"   in dir() else None,
    "kpi_top_regions":            df_kpi_top_regions if "df_kpi_top_regions" in dir() else None,
}

print("Écriture des tables Gold :")

for table_name, df in gold_tables.items():

    if df is not None:
        (
            df.withColumn("_gold_timestamp", current_timestamp())               # Ajout de l'horodatage de la création de la table Gold (traçabilité)
              .write
              .format("delta")                    
              .mode("overwrite")                  
              .option("overwriteSchema", "true")  # Mise à jour du schéma si nécessaire
              .saveAsTable(table_name)            
                                                 
        )

        print(f"  [OK] {table_name:<40} -> table managée enregistrée")

    else:

        print(f"  [SKIP] {table_name:<40} -> ignorée (données manquantes)")       # Si le DataFrame n'a pas pu être créé (colonne manquante, erreur précédente...). on journalise un avertissement mais on ne bloque pas les autres tables

Écriture des tables Gold :
  [OK] kpi_conso_par_region                     -> table managée enregistrée


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  [OK] kpi_tendance_nationale                   -> table managée enregistrée
  [OK] kpi_categorie_distribution               -> table managée enregistrée
  [OK] kpi_top_regions                          -> table managée enregistrée


## Vue d'ensemble du pipeline complet

    API Open Data Enedis
         |
         v
    BRONZE  : donnees brutes Delta Lake + metadonnees d'ingestion
         |
         v
    SILVER  : donnees nettoyées + UDFs Python + Data Quality + RGPD
         |
         v
    GOLD    : KPIs metier prets pour exploitation BI / dashboards

Compétences techniques illustrées dans ce projet :

  - Pipeline d'ingestion batch avec Delta Lake (Bronze)

  - Transformations PySpark avec detection dynamique du schema (Silver)

  - UDFs Python : categorisation energetique et score carbone (Silver)

  - Controles qualite programmatiques avec rapport consolide (Silver)

  - Conformite RGPD par k-anonymisation (Silver)

  - Window Functions Spark pour les calculs de rang et de variation YoY (Gold)

  - Architecture Lakehouse Bronze / Silver / Gold avec Delta Lake